In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((28, 28)),

    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),

    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here
#dataloader:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

#display sample

# batch from training data
import matplotlib.pyplot as plt
import numpy as np
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))

    ax.imshow(img)
    #ax.set_title(classes[labels[i].item()])
    ax.axis("off")

plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s
import torch
import torchvision.models as models
from torch import nn
from tqdm import tqdm


# Write your code here
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
# freeze
for param in model.features.parameters():
    param.requires_grad = False
#replace
in_features = model.classifier[1].in_features
# model.classifier[1] = nn.Linear(num_features, 1) # Binary classification
model.classifier[1] = nn.Linear(in_features, 26)


# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Verify what's trainable
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Training {trainable_params:,} / {total_params:,} parameters ({100*trainable_params/total_params:.2f}%)")


In [ ]:
# from tqdm import tqdm    # Shows progress bar

# # 🔹 Training Loop
# def train_one_epoch(model, dataloader, criterion, optimizer, device):
#     model.train()  # Set model to training mode
#     total_loss = 0
#     correct = 0
#     total = 0

#     for images, labels in tqdm(dataloader):
#         images, labels = images.to(device), labels.to(device)


#         outputs = model(images)  # Forward pass
#         loss = criterion(outputs, labels)  # Compute loss

#         optimizer.zero_grad()  # Reset gradients
#         loss.backward()  # Backpropagation
#         optimizer.step()  # Update weights

#         total_loss += loss.item()

In [ ]:

def train_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train() #set model to training mode
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)   # Move data to GPU if available
        labels = labels - 1 #25
        #forward pass
        optimizer.zero_grad()#reset gradients
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        #backward pass
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
    return total_loss / len(train_loader.dataset)


def validate(model, test_loader, loss_fn, device):
    model.eval() # set model to val mode
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1 #25

            outputs = model(images)
            loss = loss_fn(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(test_loader.dataset)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Write your code here
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') #setup deviceee
model = model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)

# Training loop
num_epochs = 3
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device)
    train_losses.append(train_loss)

    # Validate
    val_loss, val_acc = validate(model, test_loader, loss_fn, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, val Loss: {val_loss:.4f}, val Acc: {val_acc:.4f}")



In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss")
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# plt.plot(range(1, num_epochs+1), train_accs, label="Train Accuracy")
plt.plot(val_acc, label="Validation Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Accuracy")
plt.show()

In [ ]:
# def validate(model, test_loader, loss_fn, device):
#     model.eval() # set model to val mode
#     total_loss = 0.0
#     correct = 0
#     total = 0

#     with torch.no_grad():
#         for images, labels in test_loader:
#             images, labels = images.to(device), labels.to(device)
#             labels = labels - 1 #25

#             outputs = model(images)
#             loss = loss_fn(outputs, labels)

#             total_loss += loss.item() * images.size(0)
#             _, predicted = torch.max(outputs.data, 1)
#             total += labels.size(0)
#             correct += (predicted == labels).sum().item()

#     avg_loss = total_loss / len(test_loader.dataset)
#     accuracy = correct / total

#     return avg_loss, accuracy

In [ ]:
# Write your code here

def validate_with_tta(model, test_loader, loss_fn, device):
    model.eval() # set model to val mode
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels - 1  # 25

            #3 augmented versions
            orginal = model(images) #orginal
            horizontal = torch.flip(images, dims=[3])  # horizontal flip
            horizontal_flipped= model(horizontal)
            vertical = torch.flip(images, dims=[2])  # vertical flip
            vertical_flipped = model(vertical)

            avg_output = (orginal + horizontal_flipped + vertical_flipped) / 3

            loss = loss_fn(avg_output, labels)
            total_loss += loss.item() * images.size(0)

            _, predicted = torch.max(avg_output.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / len(test_loader.dataset)
    accuracy = correct / total

    return avg_loss, accuracy


tta_loss, tta_acc = validate_with_tta(model, test_loader, loss_fn, device)
standard_loss, standard_acc = validate(model, test_loader, loss_fn, device)

print(f"Standard Loss: {standard_loss:.4f}sStandard accuracy: {standard_acc:.4f}")
print(f" validation Loss: {tta_loss:.4f}, validation accuracy: {tta_acc:.4f}")
print(f"Accuracy with TTA: {(tta_acc - standard_acc):.4f}")